# RepairScope-X — Notebook 29C (New frozen v2): Harder Semantic-Failure Study

**Why a new protocol is necessary:** the v1 bank has been fully exhausted and cannot provide the specified **4/4/4 calibration and 8/8/8 held-out semantic failures**. All v1 records remain intact in `NB29_SemanticError_FP32_V1`. This is a **new and non-poolable experiment**, with a different task distribution, different model prompts, task IDs, source seed, bank hash, and Drive directory.

This notebook maintains real ALoDLM-1.7B FP32 inference, official model-native decoding, strict explicit wrong-answer eligibility, fixed task quotas, frozen splits, append-only checkpoints, paired problem-level analysis, and two-stage comparisons. **It cannot guarantee that the model will produce enough eligible wrong answers**. No study should turn correct boxed answers, missing/truncated answers, or unconfirmed examples into fake semantic failures.

**Running:** Fresh Colab T4. Run B0–B5. Then run B6 cells sequentially. Each cell checks for remaining failures and runs at most **eight** new generations. Once both splits reach quotas, run B7 then B8–B14. You can stop the B6 cells when both quotas are met. If you hit a session limit, rerun setup in a new session and continue; Drive checkpoints are reused. Initial model verification remains mandatory.


### B0. Official ALoDLM source / dependencies — run only AFTER runtime restart

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
REPO_DIR='/content/ALoDLM_RSX29'
if not os.path.isdir(os.path.join(REPO_DIR,'.git')):
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
print('Pinned official ALoDLM source:',CODE_COMMIT)

### B1. Real T4 GPU, new frozen protocol, and append-only Drive storage

In [ ]:
import os, sys, gc, json, csv, re, math, ast, time, random, hashlib, zipfile, tempfile, statistics, shutil, platform, traceback, subprocess
from datetime import datetime,timezone
from pathlib import Path
from collections import defaultdict,Counter
from dataclasses import dataclass
import torch, numpy as np, pandas as pd, transformers
from scipy.stats import binomtest, beta
from huggingface_hub import snapshot_download
from types import SimpleNamespace
from google.colab import drive, files
import matplotlib.pyplot as plt
from IPython.display import display
if not torch.cuda.is_available(): raise RuntimeError('REQUIRES a real GPU: Colab > Runtime > Change runtime type > T4 GPU.')
if torch.cuda.get_device_properties(0).total_memory < 12*2**30: raise RuntimeError('Needs at least 12 GiB GPU VRAM.')
if torch.cuda.memory_allocated() > 2*2**30: raise RuntimeError('Restart runtime: unexpected other GPU allocations.')
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/RepairScopeX/NB29_SemanticError_FP32_V2_Harder')
ROOT.mkdir(parents=True,exist_ok=True)
ART=ROOT/'artifacts'; ART.mkdir(exist_ok=True)
CACHE=ROOT/'records'; CACHE.mkdir(exist_ok=True)
MAX_SESSION_MINUTES=160
SESSION_START=time.monotonic()
MODEL_ID='amazon/ALoDLM-1.7B'
MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
PRECISION='fp32'; Q=0.4; TAU=0.4; WINDOW=16; MAX_NEW=96
GEN_MODE='left1'; REPAIR_MODE='entropy'
BASE_SEED=1042709
CAL_TARGET={'countdown':4,'arithmetic':4,'multistep':4}
TEST_TARGET={'countdown':8,'arithmetic':8,'multistep':8}
ACTIONS=('global','tail25','tail50','tail75')
CAL_SEEDS=(23001,23002)
TEST_SEEDS=(24001,24002,24003)
STAGED_SEEDS=(25001,25002)
ABLATION_SEEDS=(26001,26002)
POLICIES=('global','best_fixed','adaptive','random_scope','tail50')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print('Torch',torch.__version__,'Transformers',transformers.__version__)

def canonical(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'),default=str)
def sha(v): return hashlib.sha256((v if isinstance(v,bytes) else canonical(v).encode())).hexdigest()
def atomic_write(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('w',encoding='utf8') as h:
        h.write(json.dumps(obj,ensure_ascii=False,sort_keys=True,indent=2,default=str)+'\n');h.flush();os.fsync(h.fileno())
    os.replace(tmp,path)
def rec_path(stage,tid):return CACHE/stage/(sha(tid)+'.json')
def save_rec(stage,rec):
    path=rec_path(stage,rec['trial_id'])
    if path.exists():
        old=json.loads(path.read_text());
        if sha(old)!=sha(rec):raise RuntimeError('Refusing to overwrite a different completed record: '+rec['trial_id'])
        return
    atomic_write(path,rec)
def get_rec(stage,tid):
    p=rec_path(stage,tid)
    if not p.exists():return None
    r=json.loads(p.read_text())
    if r['trial_id']!=tid or r.get('protocol_hash')!=RUN_SIG:raise RuntimeError('Checkpoint signature or identity mismatch '+tid)
    return r
def all_recs(stage):
    out=[]
    for p in sorted((CACHE/stage).glob('*.json')):
        r=json.loads(p.read_text())
        assert r['protocol_hash']==RUN_SIG
        out.append(r)
    assert len({r['trial_id'] for r in out})==len(out)
    return out
def stop_if_near_budget():
    used=(time.monotonic()-SESSION_START)/60
    if used >= MAX_SESSION_MINUTES:raise RuntimeError(f'Session wall-time safety stop ({used:.1f}/{MAX_SESSION_MINUTES} min). Export partial ZIP and resume in a new Colab session; earlier records are saved.')

PROTOCOL={'version':'RSX29-AloDLM-semantic-wrong-v2-hard-composite','model':MODEL_ID,'model_rev':MODEL_REV,'code_commit':CODE_COMMIT,
          'fp':PRECISION,'q':Q,'tau':TAU,'window':WINDOW,'max_new':MAX_NEW,
          'gen_mode':GEN_MODE,'repair_mode':REPAIR_MODE,'cal_target':CAL_TARGET,'test_target':TEST_TARGET,
          'actions':ACTIONS,'cal_seeds':CAL_SEEDS,'test_seeds':TEST_SEEDS,
          'staged_seeds':STAGED_SEEDS,'ablation_seeds':ABLATION_SEEDS,'base_seed':BASE_SEED,
          'benchmark_difficulty':'predefined 5-7-operation integer tasks',
          'bank_cal_per_kind':160,'bank_heldout_per_kind':240}
RUN_SIG=sha(PROTOCOL)
manifest=ROOT/'protocol.json'
if manifest.exists():
    existing=json.loads(manifest.read_text())
    if existing['hash']!=RUN_SIG or sha(existing['protocol'])!=sha(PROTOCOL):raise RuntimeError('Different experiment in this Drive path. NEVER overwrite or mix protocols. Use a new folder.')
else:atomic_write(manifest,{'hash':RUN_SIG,'protocol':PROTOCOL,'created_utc':datetime.now(timezone.utc).isoformat()})

### Optional: immediate partial export at any time (safe even without a frozen cohort)
If the GPU quota is interrupted, execute this short cell to save a checksum-verified snapshot. It does not claim completion. For the normal final export, use B14 after analysis.

In [ ]:
def export_checkpoint_snapshot(download=False):
    import zipfile, hashlib, json
    from pathlib import Path
    out=Path('/content/RepairScopeX_NB29_Partial_Checkpoint.zip')
    paths=[]
    for name in ('protocol.json','frozen_problem_bank.json','frozen_cohorts.json','FROZEN_POLICY.json'):
        p=ROOT/name
        if p.exists(): paths.append((p,'protocol/'+name))
    for parent,label in ((CACHE,'records'),(ART,'artifacts')):
        if parent.exists():
            for p in parent.rglob('*'):
                if p.is_file(): paths.append((p,label+'/'+str(p.relative_to(parent))))
    paths.sort(key=lambda pair:pair[1]); checks={name:hashlib.sha256(p.read_bytes()).hexdigest() for p,name in paths}
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        z.writestr('PARTIAL_STATUS.json',json.dumps({'partial':True,'protocol_hash':RUN_SIG,'sha256':checks},indent=2))
        for p,name in paths:z.write(p,name)
    with zipfile.ZipFile(out) as z:
        assert z.testzip() is None
        for name,sig in checks.items():assert hashlib.sha256(z.read(name)).hexdigest()==sig
    print('Safe checkpoint snapshot:',out,'files:',len(paths))
    if download:
        from google.colab import files
        files.download(str(out))
    return out
print('Optional checkpoint snapshot available: export_checkpoint_snapshot(download=True)')

### B2. Disjoint, frozen deterministic benchmark (no model-generated gold labels)

In [ ]:
# RSX29 v2: independent, deterministic composite-integer benchmark.
# Its prompts, answers, splits and order are frozen BEFORE any v2 GPU outputs are inspected.
# V1 data are preserved in their independent, unmodified directory.

def make_problem(kind,rng,i,split):
    if kind=='countdown':
        a,b,c=[rng.randint(6,55) for _ in range(3)]
        mode=i%6
        if mode==0: target=a+b-c; expr=f'({a}+{b}-{c})'
        elif mode==1: target=a*b-c; expr=f'({a}*{b}-{c})'
        elif mode==2: target=a*(b-c); expr=f'({a}*({b}-{c}))'
        elif mode==3: target=a*b+c; expr=f'({a}*{b}+{c})'
        elif mode==4: target=(a+b)*c; expr=f'(({a}+{b})*{c})'
        else: target=(a-b)*c; expr=f'(({a}-{b})*{c})'
        if target<=0: raise ValueError('Nonpositive Countdown target; generate a new seeded task.')
        prompt=(f'Countdown puzzle: use exactly the three numbers {a}, {b}, {c} once each '
                f'with +, -, *, / and parentheses to obtain {target}. Give your expression '
                f'and finish with FINAL_ANSWER: <number>.')
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[a,b,c],
                'template':f'countdown_{mode}','gold_expression':expr}

    if kind=='arithmetic':
        a=rng.randint(34,175);b=rng.randint(13,93);c=rng.randint(17,124)
        d=rng.randint(9,57);e=rng.randint(5,88);f=rng.randint(4,79)
        g=rng.randint(11,155);h=rng.randint(7,35)
        mode=i%8
        if mode==0: expr=f'({a}*{b}+{c}*{d}-{e}*{f}+{g})'
        elif mode==1: expr=f'(({a}*{b}-{c})*{d}+{e}*{f})'
        elif mode==2: expr=f'(({a}+{b})*{c}-{d}*{e}+{f}*{g})'
        elif mode==3: expr=f'({a}*{b}*{d}-{c}*{e}+{f}-{g})'
        elif mode==4: expr=f'(({a}*{b}+{c}*{d})-{e}*{f}-{g})'
        elif mode==5: expr=f'(({a}+{b}-{c})*{d}+{e}*{f}-{g})'
        elif mode==6: expr=f'({a}*({b}+{c})-{d}*({e}+{f})+{g})'
        else: expr=f'(({a}*{b}-{c}*{d}+{e})*{f}-{g}*{h})'
        target=safe_gold_eval(expr)
        prompt=(f'Compute this integer arithmetic expression precisely, respecting parentheses '
                f'and operator precedence: {expr}. Give only a brief calculation and end with '
                f'FINAL_ANSWER: <number>.')
        return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],
                'template':f'composite_arith_{mode}','gold_expression':expr}

    # Every word problem requires 5 or more integer operations; no rounding or outside knowledge.
    a=rng.randint(18,97);b=rng.randint(13,79);c=rng.randint(9,58)
    d=rng.randint(7,69);e=rng.randint(4,37);f=rng.randint(5,28)
    g=rng.randint(4,68);h=rng.randint(9,47);j=rng.randint(6,35)
    mode=i%8
    if mode==0:
        expr=f'({a}*{b}+{c}*{d}-{e}*{f}-{g})'
        prompt=(f'A warehouse starts with {a} cases containing {b} items per case. '
                f'It receives {c} extra cases containing {d} items per case. '
                f'Then it ships {e} cases containing {f} items per case and discards '
                f'{g} damaged items. How many items are left?')
    elif mode==1:
        expr=f'({a}*{b}+{c}*{d}-{e}*{f}-{g})'
        prompt=(f'A theater sells {a} premium tickets at {b} dollars each and '
                f'{c} standard tickets at {d} dollars each. It refunds {e} tickets '
                f'worth {f} dollars each and pays {g} dollars in fees. What is '
                f'its net ticket revenue in dollars?')
    elif mode==2:
        expr=f'(({a}+{b})*{c}-{d}*{e}+{f}*{g})'
        prompt=(f'A school has {a} shelves of science books and {b} shelves of '
                f'math books. Every shelf holds {c} books. Students borrow '
                f'{d} bundles of {e} books, then donors bring {f} cartons of '
                f'{g} books. How many books are in the school afterward?')
    elif mode==3:
        expr=f'({a}*{b}*{c}-{d}*{e}-{f}*{g})'
        prompt=(f'A plant has {a} machines. Each makes {b} parts per shift for '
                f'{c} shifts. Quality control discards {d} batches of {e} parts. '
                f'Another {f} boxes containing {g} parts each are shipped. '
                f'How many manufactured parts remain?')
    elif mode==4:
        expr=f'(({a}*{b}+{c}*{d})-{e}*{f}+{g}*{h})'
        prompt=(f'A farm packs {a} crates with {b} oranges each and {c} bags with '
                f'{d} oranges each. It sells {e} orders of {f} oranges and '
                f'harvests {g} more baskets of {h} oranges. How many oranges '
                f'does it have now?')
    elif mode==5:
        expr=f'({a}*({b}+{c})-{d}*{e}-{f}*{g})'
        prompt=(f'A training center has {a} rooms, each with {b} existing chairs. '
                f'It adds {c} chairs in every room. Then it sends away {d} '
                f'groups of {e} chairs and repairs remove another {f} groups '
                f'of {g}. How many chairs remain in service?')
    elif mode==6:
        expr=f'(({a}*{b}-{c}*{d})*{e}+{f}*{g})'
        prompt=(f'Each of {e} independent regional depots begins with {a} cartons, '
                f'with {b} packets in each carton. Each depot returns '
                f'{c} cartons holding {d} packets each. A central depot then '
                f'receives another {f} crates of {g} packets. How many '
                f'packets remain across all depots?')
    else:
        expr=f'(({a}+{b})*{c}+{d}*{e}-{f}*{g}-{h}*{j})'
        prompt=(f'Two branches each report the following number of orders: '
                f'the first has {a}, the second {b}. Every order contains '
                f'{c} products. Another branch sends {d} boxes of {e} products. '
                f'The company returns {f} boxes of {g} products and discards '
                f'{h} groups of {j} products. How many products remain?')
    target=safe_gold_eval(expr)
    if target<0 or (mode==6 and a*b<c*d):
        raise ValueError('Impossible negative inventory; use next seeded candidate.')
    prompt += ' Work the numbers out carefully and end with FINAL_ANSWER: <number>.'
    return {'kind':kind,'prompt':prompt,'target':float(target),'numbers':[],
            'template':f'multistep_{mode}','gold_expression':expr}

def safe_gold_eval(expr):
    # Independent, safe evaluator with exact integer math for synthetic gold generation.
    import ast
    tree=ast.parse(expr,mode='eval')
    def ev(n):
        if isinstance(n,ast.Expression):return ev(n.body)
        if isinstance(n,ast.Constant) and type(n.value) is int:return n.value
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,ast.USub):return -ev(n.operand)
        if isinstance(n,ast.BinOp):
            l,r=ev(n.left),ev(n.right)
            if isinstance(n.op,ast.Add):return l+r
            if isinstance(n.op,ast.Sub):return l-r
            if isinstance(n.op,ast.Mult):return l*r
        raise ValueError('Unsafe or non-integer gold expression')
    return ev(tree)

GLOBAL_FINGERPRINTS=set()
def build_bank(split,n_per_kind):
    out=[];seen=set()
    for k_idx,kind in enumerate(('countdown','arithmetic','multistep')):
        for i in range(n_per_kind):
            for retry in range(300):
                seed=BASE_SEED+(0 if split=='calibration' else 1000000)+k_idx*10000+i*11+retry*310009
                rng=random.Random(seed)
                try:
                    d=make_problem(kind,rng,i,split)
                except ValueError:
                    continue
                fp=sha({'prompt':d['prompt'],'kind':kind})
                if fp not in seen and fp not in GLOBAL_FINGERPRINTS:
                    seen.add(fp);GLOBAL_FINGERPRINTS.add(fp)
                    d.update(split=split,problem_id=f'rsx29v2:{split}:{kind}:{i:04d}',
                             problem_fingerprint=fp,collision_retry=retry)
                    out.append(d)
                    break
            else:raise RuntimeError('Could not generate a unique deterministic task.')
    return out
CAL_BANK=build_bank('calibration',160)
TEST_BANK=build_bank('heldout',240)
assert len(CAL_BANK)==480 and len(TEST_BANK)==720
assert len(set(r['problem_fingerprint'] for r in CAL_BANK+TEST_BANK))==len(CAL_BANK+TEST_BANK)
for p in CAL_BANK+TEST_BANK:
    assert float(p['target']).is_integer()
    assert safe_gold_eval(p['gold_expression'])==int(p['target'])
    assert 'FINAL_ANSWER:' in p['prompt']
BANK_HASH=sha(CAL_BANK+TEST_BANK)
BANK_PATH=ROOT/'frozen_problem_bank.json'
if BANK_PATH.exists():
    assert json.loads(BANK_PATH.read_text())['hash']==BANK_HASH, 'Bank mismatch: stop, do not overwrite.'
else:atomic_write(BANK_PATH,{'hash':BANK_HASH,'problems':CAL_BANK+TEST_BANK})
print('FROZEN V2 BANK: 480 calibration and 720 heldout tasks; sha256=',BANK_HASH)


### B3. STRICT semantic-error eligibility and answer-format separation

In [ ]:
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
NUM_RE=re.compile(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?')
RUN_RE=re.compile(r'[0-9()+*×÷/\-+=\s]{3,}')
def closenum(a,b):return a is not None and b is not None and math.isclose(float(a),float(b),rel_tol=1e-9,abs_tol=1e-8)
def safe_expression(s):
    try:t=ast.parse(s,mode='eval')
    except Exception:return None
    lit=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (int,float):lit.append(float(n.value));return float(n.value)
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            a=walk(n.operand);return (-a if isinstance(n.op,ast.USub) else a)
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            if isinstance(n.op,ast.Add):return a+b
            if isinstance(n.op,ast.Sub):return a-b
            if isinstance(n.op,ast.Mult):return a*b
            if abs(b)<1e-12:raise ZeroDivisionError
            return a/b
        raise ValueError('Unsupported expression')
    try:
        v=walk(t)
        if math.isfinite(v):return {'value':v,'literals':lit}
    except Exception:pass
    return None

def extract_expression(text,problem):
    cleaned=text.replace('×','*').replace('÷','/')
    for m in RUN_RE.finditer(cleaned):
        raw=m.group(0)
        # Restrict to expression strings and avoid accidentally matching a target alone.
        for part in raw.split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            result=safe_expression(ex)
            if not result:continue
            if len(result['literals'])==len(problem['numbers']) and sorted(result['literals'])==sorted(map(float,problem['numbers'])) and closenum(result['value'],problem['target']):return ex
    return None

BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')

def last_explicit_number(text):
    """Only explicit answer markers, never final arbitrary numerals in unfinished reasoning.
    Conflicting explicit answer values => ambiguous, never an eligible failure."""
    text=str(text)
    entries=[]
    for pattern,kind in ((FINAL_RE,'final'),(BOX_RE,'boxed'),(ANSWER_RE,'answer_phrase')):
        for m in pattern.finditer(text):
            try:entries.append((m.start(),float(m.group(1)),kind))
            except (ValueError,OverflowError):pass
    if not entries:return None,'missing'
    unique=[]
    for _,v,_ in entries:
        if not any(closenum(v,w) for w in unique):unique.append(v)
    if len(unique)>1:return None,'conflicting_markers'
    entries.sort()
    return entries[-1][1],entries[-1][2]

def countdown_wrong_candidate(text,p):
    """Inspectable wrong *complete* expression consuming exactly the supplied numbers.
    Do not equate missing expressions with semantic error."""
    for m in RUN_RE.finditer(str(text).replace('×','*').replace('÷','/')):
        for part in m.group(0).split('='):
            ex=part.strip()
            if not any(ch in ex for ch in '+-*/'):continue
            r=safe_expression(ex)
            if r and len(r['literals'])==len(p['numbers']) and sorted(r['literals'])==sorted(map(float,p['numbers'])):
                if not closenum(r['value'],p['target']):return ex
    return None

def grade_output(text,p):
    text=str(text)
    num,origin=last_explicit_number(text)
    sem_numeric=closenum(num,p['target'])
    last_nonempty=next((line.strip() for line in reversed(text.splitlines()) if line.strip()),'')
    fmt=FINAL_RE.fullmatch(last_nonempty)
    strict=bool(fmt and closenum(float(fmt.group(1)),p['target']))
    if p['kind']=='countdown':
        expr=extract_expression(text,p)
        success=expr is not None
        wrong_expr=None if success else countdown_wrong_candidate(text,p)
        explicit_wrong=(num is not None and not sem_numeric)
        # An explicitly wrong answer or an inspectable illegal computation, with no valid solution.
        is_wrong=bool(not success and origin!='conflicting_markers' and (explicit_wrong or wrong_expr is not None))
    else:
        expr=None;wrong_expr=None
        success=bool(sem_numeric)
        is_wrong=bool(num is not None and not sem_numeric and origin!='conflicting_markers')
    return {'success':bool(success),'numeric_correct':bool(sem_numeric),
            'strict_correct':bool(strict),'valid_expression':expr,
            'wrong_expression':wrong_expr,'answer_origin':origin,
            'explicit_final_number':num,'eligible_semantic_failure':bool(is_wrong),
            'failure_type':('definite_wrong' if is_wrong else
                   ('semantically_correct' if success else 'unanswered_or_ambiguous'))}

def public_accept(text,p):
    """Label-blind stopping gate: never consult word-problem gold targets."""
    if p['kind']=='countdown':
        # Target is explicitly present in the Countdown input; a valid expression may be verified.
        return extract_expression(str(text),p) is not None
    num,origin=last_explicit_number(str(text))
    return num is not None and origin!='conflicting_markers'

def features(text,p,n_tokens):
    """Target-free features; no test labels or correctness queries."""
    num,origin=last_explicit_number(text)
    L=int(n_tokens)
    return {'kind':p['kind'],'length_bucket':'short' if L<45 else ('medium' if L<80 else 'long'),
            'has_final':bool(FINAL_RE.search(str(text))), 'answer_kind':origin,
            'repetition':bool(re.search(r'(\b\w+\b)(?:\s+\1){2,}',str(text),re.I))}

def feat_bucket(f):return f['kind']+'|'+('final' if f['has_final'] else 'no_final')

# Essential semantic-failure regression checks (no GPU or network needed):
q={'kind':'arithmetic','target':14,'numbers':[]}
assert grade_output(r'We have \boxed{14}.',q)['success']
assert not grade_output(r'We have \boxed{14}.',q)['eligible_semantic_failure']
assert grade_output('FINAL_ANSWER: 13',q)['eligible_semantic_failure']
assert not grade_output('Then 14 times',q)['eligible_semantic_failure']
assert not grade_output('No answer yet',q)['eligible_semantic_failure']
assert not grade_output('FINAL_ANSWER: 13\n\\boxed{14}',q)['eligible_semantic_failure']
assert grade_output('The answer is 14',q)['success']
p={'kind':'countdown','numbers':[2,3,4],'target':14}
assert grade_output('2 + 3 * 4 = 14',p)['success']
assert not grade_output('2 + 3 * 4 = 14',p)['eligible_semantic_failure']
assert grade_output('2 + 3 + 4 = 9',p)['eligible_semantic_failure']
assert not grade_output('14',p)['eligible_semantic_failure']
assert not grade_output('2 + 4 + 4 = 14',p)['success']
print('Semantic grader regression suite PASSED: correct boxed excluded, explicit wrong admitted, truncations rejected.')


### B4. Load official ALoDLM FP32 model; real CUDA smoke test

In [ ]:
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
assert MODEL_REV in str(MODEL_DIR) or Path(MODEL_DIR).is_dir()
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
decoder=load_decoder(args)
model=decoder.model;tokenizer=decoder.tokenizer
assert next(model.parameters()).dtype==torch.float32
print('Model loaded FP32; CUDA allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
PROMPT_SUFFIX='\n\nIf helpful, calculate briefly. For Countdown show an expression using all numbers exactly once. End with FINAL_ANSWER: <number>.'
def encode_prompt(p):
    return list(tokenizer.apply_chat_template([{'role':'user','content':p['prompt']+PROMPT_SUFFIX}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
def decode_ids(ids):return tokenizer.decode(ids,skip_special_tokens=True).strip()
def set_seed(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
@torch.inference_mode()
def invoke(prompt_ids,new_tokens,mode,seed):
    assert new_tokens>0 and new_tokens<=MAX_NEW
    stop_if_near_budget();set_seed(int(seed))
    cfg=DecodeConfig(q=Q,tau=TAU,mode=mode,window_size=WINDOW,max_new_tokens=int(new_tokens),position_penalty=None)
    cfg.validate()
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter();result=decoder.generate(list(prompt_ids),cfg)
    ids=list(map(int,result['token_ids']))
    return {'token_ids':ids,'generated_tokens':int(result.get('generated_tokens',len(ids))),
            'outer_steps':int(result['outer_steps']),'recurrent_passes':int(result['recurrent_passes']),
            'loop_token_passes':int(result['loop_token_passes']),
            'mean_exit_depth':float(result['mean_exit_depth']) if result.get('mean_exit_depth') is not None else None,
            'wall_seconds':float(time.perf_counter()-started),'internal_wall_seconds':float(result.get('wall_seconds',0)),
            'stop_reason':str(result.get('stop_reason','unknown')),
            'peak_vram_gb':float(torch.cuda.max_memory_allocated()/2**30)}
smoke=invoke(encode_prompt(CAL_BANK[0]),12,GEN_MODE,BASE_SEED)
assert smoke['outer_steps']>0 and smoke['loop_token_passes']>0
print('REAL CUDA smoke PASSED', {k:smoke[k] for k in ('outer_steps','loop_token_passes','wall_seconds','peak_vram_gb')})

### B5. Model-native semantic-failure discovery (correct boxed answers excluded)

In [ ]:
def natural_id(p):return p['problem_id']+'|natural|'+str(BASE_SEED)
def natural_run(p):
    tid=natural_id(p);old=get_rec('natural',tid)
    if old:return old
    r=invoke(encode_prompt(p),MAX_NEW,GEN_MODE,BASE_SEED)
    txt=decode_ids(r['token_ids']);v=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':p['problem_id'],'split':p['split'],
         'kind':p['kind'],'problem_fingerprint':p['problem_fingerprint'],'original_output':txt,
         'original_token_ids':r.pop('token_ids'),'evaluation':v,**r}
    save_rec('natural',obj)
    return obj

def cohort_from_natural(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    bykind=defaultdict(list)
    for p in bank:
        rec=get_rec('natural',natural_id(p))
        if rec and rec['evaluation']['eligible_semantic_failure']:bykind[p['kind']].append(p['problem_id'])
    if not all(len(bykind[k])>=targets[k] for k in targets):
        print('Eligible failures:',{k:len(bykind[k]) for k in targets},'required:',targets)
        return None
    subset=[x for k in targets for x in bykind[k][:targets[k]]]
    return subset

def natural_batch(split,start,limit=8):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    if cohort_from_natural(split):print('Already enough failures; skipping further candidate generation.');return
    for p in bank[start:start+limit]:
        if cohort_from_natural(split):break
        r=natural_run(p)
        print(p['problem_id'],r['evaluation']['success'],round(r['wall_seconds'],2),'s')
    print('Progress',len(all_recs('natural')),'native generations total.')

def freeze_cohorts():
    cp=cohort_from_natural('calibration');hp=cohort_from_natural('heldout')
    assert cp is not None and hp is not None,'Not enough model-native failures yet: continue natural-generation batches.'
    assert set(cp).isdisjoint(hp)
    content={'protocol_hash':RUN_SIG,'calibration':cp,'heldout':hp,'problem_bank_sha256':BANK_HASH}
    file=ROOT/'frozen_cohorts.json'
    if file.exists():assert json.loads(file.read_text())==content,'Frozen cohort differs; refusing test-set mutation.'
    else:atomic_write(file,content)
    print('FROZEN',len(cp),'calibration and',len(hp),'heldout native failures.')
    return content

def v2_cohort_progress(split):
    bank=CAL_BANK if split=='calibration' else TEST_BANK
    targets=CAL_TARGET if split=='calibration' else TEST_TARGET
    eligible={k:0 for k in targets}; unused={k:0 for k in targets}; pending=[]
    for p in bank:
        r=get_rec('natural',natural_id(p))
        if r is None:
            unused[p['kind']]+=1;pending.append(p)
        elif r['evaluation']['eligible_semantic_failure']:
            eligible[p['kind']]+=1
    missing={k:max(0,targets[k]-eligible[k]) for k in targets}
    return eligible,missing,unused,pending

def v2_discovery_batch(split,n=8):
    """Runs at most n new GPU calls, prioritizing only currently unmet task types."""
    eligible,missing,unused,pending=v2_cohort_progress(split)
    todo=[p for p in pending if missing[p['kind']]>0][:n]
    if not todo:
        print(split,'NO CANDIDATES TO RUN; eligible',eligible,'missing',missing,'remaining',unused)
        return
    for p in todo:
        stop_if_near_budget()
        r=natural_run(p)
        print(p['problem_id'],'failure_type=',r['evaluation']['failure_type'],
              'eligible=',r['evaluation']['eligible_semantic_failure'],
              'seconds=',round(r['wall_seconds'],2))
    eligible,missing,unused,_=v2_cohort_progress(split)
    print(split,'eligible=',eligible,'missing=',missing,'unused=',unused)
    if any(missing[k] and not unused[k] for k in missing):
        print('V2 BANK EXHAUSTED. Export partial results; do not weaken eligibility.')



### B6.1 Calibration candidates — prioritized, eight GPU calls maximum per cell
Run until quotas are reached; if the bank is exhausted, export the partial audit and stop.

In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 1/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 2/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 3/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 4/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 5/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 6/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 7/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 8/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 9/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 10/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 11/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 12/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 13/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 14/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 15/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 16/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 17/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 18/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 19/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 20/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 21/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 22/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 23/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 24/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 25/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 26/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 27/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 28/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 29/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 30/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 31/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 32/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 33/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 34/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 35/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 36/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 37/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 38/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 39/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 40/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 41/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 42/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 43/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 44/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 45/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 46/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 47/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 48/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 49/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 50/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 51/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 52/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 53/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 54/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 55/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 56/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 57/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 58/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 59/60


In [ ]:
v2_discovery_batch('calibration',8)  # calibration generation batch 60/60


In [ ]:
print('Calibration counts:',v2_cohort_progress('calibration')[:3])


### B6.2 Held-out candidates — separately frozen, no policy tuning on test outcomes
Run the next batch only if missing quota remains, never select based on held-out repair performance.

In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 1/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 2/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 3/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 4/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 5/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 6/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 7/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 8/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 9/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 10/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 11/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 12/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 13/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 14/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 15/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 16/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 17/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 18/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 19/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 20/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 21/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 22/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 23/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 24/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 25/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 26/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 27/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 28/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 29/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 30/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 31/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 32/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 33/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 34/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 35/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 36/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 37/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 38/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 39/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 40/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 41/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 42/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 43/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 44/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 45/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 46/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 47/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 48/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 49/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 50/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 51/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 52/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 53/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 54/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 55/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 56/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 57/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 58/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 59/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 60/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 61/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 62/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 63/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 64/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 65/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 66/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 67/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 68/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 69/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 70/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 71/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 72/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 73/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 74/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 75/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 76/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 77/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 78/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 79/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 80/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 81/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 82/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 83/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 84/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 85/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 86/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 87/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 88/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 89/90


In [ ]:
v2_discovery_batch('heldout',8)  # heldout generation batch 90/90


In [ ]:
print('Heldout counts:',v2_cohort_progress('heldout')[:3])


### B7. Freeze v2 semantic cohorts. If not enough WRONG answers, do not proceed.

In [ ]:
# REPLACE the entire B7 cell (COHORTS=freeze_cohorts() ...) with this.
# Requires the earlier setup, frozen bank, verifier, and model cells to have run.
# Safe to rerun. Generates at most 4 new examples per split per execution.

NEW_PER_SPLIT = 8

def cohort_progress(split):
    bank = CAL_BANK if split == 'calibration' else TEST_BANK
    targets = CAL_TARGET if split == 'calibration' else TEST_TARGET
    counts = {kind: 0 for kind in targets}
    pending = []
    for problem in bank:
        rec = get_rec('natural', natural_id(problem))
        if rec is None:
            pending.append(problem)
        elif rec['evaluation']['eligible_semantic_failure']:
            counts[problem['kind']] += 1
    missing = {kind: max(0, targets[kind] - counts[kind]) for kind in targets}
    remaining = {kind: sum(p['kind'] == kind for p in pending) for kind in targets}
    return counts, missing, remaining, pending

# Prioritize the task types with unmet quotas. Do not change labels, targets,
# frozen problem bank, or split assignments.
for split in ('calibration', 'heldout'):
    counts, missing, remaining, pending = cohort_progress(split)
    print(f'\n{split} BEFORE: eligible={counts}; missing={missing}; unrun={remaining}')
    if not any(missing.values()):
        continue
    todo = [p for p in pending if missing[p['kind']] > 0][:NEW_PER_SPLIT]
    for p in todo:
        r = natural_run(p)  # saves each completed generation immediately
        print('Ran:', p['problem_id'], '| eligible:',
              bool(r['evaluation']['eligible_semantic_failure']))
    counts, missing, remaining, _ = cohort_progress(split)
    print(f'{split} AFTER: eligible={counts}; missing={missing}; unrun={remaining}')

cal_counts, cal_missing, cal_remaining, _ = cohort_progress('calibration')
test_counts, test_missing, test_remaining, _ = cohort_progress('heldout')

if not any(cal_missing.values()) and not any(test_missing.values()):
    COHORTS = freeze_cohorts()  # original fingerprint and split checks remain
    PROBLEMS = {p['problem_id']: p for p in CAL_BANK + TEST_BANK}
    def failure(pid):
        p = PROBLEMS[pid]
        r = get_rec('natural', natural_id(p))
        assert (r is not None and
                r['evaluation']['eligible_semantic_failure'] and
                r['problem_fingerprint'] == p['problem_fingerprint'])
        return p, r
    print('COHORTS FROZEN (V2). Continue with Section B8.')
else:
    COHORTS = None
    exhausted = [
        (split, kind, missing[kind])
        for split, missing, remaining in (
            ('calibration', cal_missing, cal_remaining),
            ('heldout', test_missing, test_remaining))
        for kind in missing
        if missing[kind] > 0 and remaining[kind] == 0
    ]
    if exhausted:
        print('\nBANK EXHAUSTED for required semantic failures:', exhausted)
        print('This protocol cannot meet its preregistered quotas. Keep the saved'
              ' records; do NOT relabel correct boxed answers, cross splits, or'
              ' silently lower target counts. Another separately versioned follow-up'
              ' protocol with harder problems or different sampling is required.')
    else:
        print('\nNot enough strict semantic failures yet. RERUN ONLY THIS CELL'
              ' to generate another small checkpointed batch. Do not run B8 yet.')


### B8. Prefix-preserving ALoDLM real repair calls; stop-on-error checkpoints

In [ ]:
def scope_cut(n,action):
    if action=='global':return 0
    if action not in ACTIONS:raise KeyError(action)
    frac={'tail25':.75,'tail50':.50,'tail75':.25}[action]
    return max(1,min(n-1,int(math.floor(n*frac))))

def repair_one(pid,action,seed,mode=REPAIR_MODE,stage='calibration'):
    assert action in ACTIONS
    tid=f'{stage}|{pid}|{action}|{mode}|{seed}'
    old=get_rec(stage,tid)
    if old:return old
    p,source=failure(pid)
    ids=source['original_token_ids']; n=len(ids);assert n>=4
    cut=scope_cut(n,action);preserved=ids[:cut];budget=max(1,n-cut)
    assert budget<=MAX_NEW
    r=invoke(encode_prompt(p)+preserved,budget,mode,seed)
    new_full=preserved+r['token_ids']
    assert new_full[:cut]==preserved, 'Prefix-preservation invariant failed.'
    txt=decode_ids(new_full)
    eval_=grade_output(txt,p)
    obj={'trial_id':tid,'protocol_hash':RUN_SIG,'stage':stage,'problem_id':pid,'split':p['split'],
        'kind':p['kind'],'action':action,'mode':mode,'seed':int(seed),'original_response_tokens':n,
        'preserved_tokens':cut,'requested_new_tokens':budget,'preserved_fraction':cut/n,
        'exact_prefix_preserved':True,'original_output':source['original_output'],'repaired_output':txt,
        'evaluation':eval_,'public_accept':public_accept(txt,p),
        'original_repeated_exactly':txt==source['original_output'],
        **{k:v for k,v in r.items() if k!='token_ids'}}
    save_rec(stage,obj)
    return obj

def run_grid_batch(stage,grid,start,count=8):
    if grid is None:
        print('SKIPPED: calibration cohort is not frozen. Run B7 until COHORTS FROZEN, then rerun B8.')
        return False
    for pid,action,mode,seed in grid[start:start+count]:
        r=repair_one(pid,action,seed,mode,stage)
        print(f'{stage} {r["problem_id"]} {action} {mode} {seed}: success={r["evaluation"]["success"]} LTP={r["loop_token_passes"]}')
    print('Saved',len(all_recs(stage)),'stage records')

# Only construct the fixed calibration grid after B7 has actually frozen both cohorts.
# Do not substitute an empty or reduced grid: that would change the experiment.
COHORTS = globals().get('COHORTS')
if not isinstance(COHORTS, dict) or not all(k in COHORTS for k in ('calibration', 'heldout')):
    CAL_GRID = None
    print('\nB8 NOT READY: No frozen calibration/held-out cohorts yet.')
    try:
        for split in ('calibration', 'heldout'):
            counts, missing, remaining, _ = cohort_progress(split)
            print(f'{split}: eligible={counts} | missing={missing} | unused candidates={remaining}')
            if any(missing[k] and remaining[k] == 0 for k in missing):
                print('BANK EXHAUSTED in this split. Do not change or fabricate labels. '
                      'A new frozen protocol with more/harder tasks is required.')
    except NameError:
        print('Run the corrected B7 cell once first to define cohort_progress().')
    print('ACTION: Return to B7. Rerun B7 until it prints COHORTS FROZEN; '
          'then rerun B8 and proceed to B9. Do NOT run B9 yet.')
else:
    assert len(COHORTS['calibration']) == sum(CAL_TARGET.values()) == 12, 'Calibration quota mismatch.'
    assert len(COHORTS['heldout']) == sum(TEST_TARGET.values()) == 24, 'Held-out quota mismatch.'
    assert set(COHORTS['calibration']).isdisjoint(COHORTS['heldout']), 'Split overlap.'
    frozen_path = ROOT / 'frozen_cohorts.json'
    assert frozen_path.exists(), 'Cohort exists in memory but frozen checkpoint is missing.'
    frozen_saved = json.loads(frozen_path.read_text())
    assert frozen_saved == COHORTS, 'Cohort differs from saved frozen checkpoint.'
    CAL_GRID = [(pid, a, REPAIR_MODE, s)
                for pid in COHORTS['calibration'] for a in ACTIONS for s in CAL_SEEDS]
    assert len(CAL_GRID) == 96, 'Calibration trial grid size mismatch.'
    assert len(set(CAL_GRID)) == len(CAL_GRID), 'Duplicate calibration trials.'
    print('B8 READY: frozen cohorts verified; 96 calibration repairs scheduled. Proceed to B9.')


### B9. Calibrate on the 12 calibration failures (96 repair proposals)

In [ ]:
run_grid_batch('calibration',CAL_GRID,0,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,4,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,8,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,12,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,16,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,20,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,24,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,28,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,32,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,36,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,40,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,44,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,48,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,52,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,56,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,60,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,64,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,68,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,72,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,76,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,80,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,84,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,88,4)

In [ ]:
run_grid_batch('calibration',CAL_GRID,92,4)

### B10. Fit and freeze a policy using calibration outcomes ONLY

In [ ]:
def calibrate():
    data=all_recs('calibration');exp={f'calibration|{pid}|{a}|{REPAIR_MODE}|{seed}' for pid in COHORTS['calibration'] for a in ACTIONS for seed in CAL_SEEDS}
    assert len(data)==96 and set(r['trial_id'] for r in data)==exp,'Calibration incomplete; DO NOT inspect heldout results or fit policy yet.'
    pools=defaultdict(list)
    for r in data:
        p,src=failure(r['problem_id'])
        f=features(src['original_output'],p,len(src['original_token_ids']))
        pools[(feat_bucket(f),r['action'])].append(r)
        pools[(p['kind'],r['action'])].append(r)
        pools[('all',r['action'])].append(r)
    # Predeclared Laplace prior and lightweight cost penalty. No target-label features at deployment.
    def value(group,action):
        rows=pools.get((group,action),[])
        if not rows:return None
        succ=sum(r['evaluation']['success'] for r in rows)
        ltp=statistics.mean(r['loop_token_passes'] for r in rows)
        glob=pools[('all','global')]
        relative_cost=ltp/max(1,statistics.mean(r['loop_token_passes'] for r in glob))
        return (succ+1)/(len(rows)+2)-0.05*relative_cost
    fallback=max(ACTIONS,key=lambda a:(value('all',a),-ACTIONS.index(a)))
    kinds=sorted({p['kind'] for p in CAL_BANK})
    mapping={}
    for kind in kinds:
        for has_final in (True,False):
            buck=kind+'|'+('final' if has_final else 'no_final')
            group=buck if min(len(pools[(buck,a)]) for a in ACTIONS)>=6 else kind
            mapping[buck]=max(ACTIONS,key=lambda a:(value(group,a),-ACTIONS.index(a)))
    calib_summary={a:{'trials':len(pools[('all',a)]),'successes':sum(r['evaluation']['success'] for r in pools[('all',a)]),
                      'mean_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in pools[('all',a)])} for a in ACTIONS}
    frozen={'protocol_hash':RUN_SIG,'calibration_trial_hash':sha(sorted((r['trial_id'],r['evaluation']['success']) for r in data)),
            'best_fixed':fallback,'mapping':mapping,'lambda_cost':0.05,'action_summary':calib_summary,'trained_only_on':'calibration'}
    path=ROOT/'FROZEN_POLICY.json'
    if path.exists():assert json.loads(path.read_text())==frozen,'Cannot refit a different policy after freezing.'
    else:atomic_write(path,frozen)
    return frozen
POLICY=calibrate()
print('FROZEN POLICY:',json.dumps(POLICY,indent=2))

### B11. Held-out policy evaluations: 24 semantic failures × 5 policies × 3 seeds = 360 logical outcomes
Five policies share generated candidates when action+seed match. All unique GPU primitives are logged separately.

In [ ]:
def choose_action(policy,pid,seed):
    p,src=failure(pid)
    if policy=='global':return 'global'
    if policy=='tail50':return 'tail50'
    if policy=='best_fixed':return POLICY['best_fixed']
    if policy=='adaptive':
        f=features(src['original_output'],p,len(src['original_token_ids']))
        return POLICY['mapping'][feat_bucket(f)]
    if policy=='random_scope':
        # Precommitted randomization independent of targets/outcomes; includes global.
        seed_int=int(sha({'pid':pid,'seed':seed,'policy':'random_scope'})[:12],16)
        return random.Random(seed_int).choice(ACTIONS)
    raise KeyError(policy)
def heldout_one(pid,policy,seed):
    action=choose_action(policy,pid,seed)
    primitive=repair_one(pid,action,seed,REPAIR_MODE,stage='test_primitives')
    tid=f'heldout|{pid}|{policy}|{seed}'
    old=get_rec('heldout',tid)
    if old:return old
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'policy':policy,'action':action,'seed':seed,
       'primitive_id':primitive['trial_id'],'evaluation':primitive['evaluation'],'loop_token_passes':primitive['loop_token_passes'],
       'recurrent_passes':primitive['recurrent_passes'],'wall_seconds':primitive['wall_seconds'],
       'preserved_fraction':primitive['preserved_fraction'],'public_accept':primitive['public_accept'],
       'original_repeated_exactly':primitive['original_repeated_exactly']}
    save_rec('heldout',r)
    return r
TEST_GRID=[(pid,pol,s) for pid in COHORTS['heldout'] for pol in POLICIES for s in TEST_SEEDS]
assert len(TEST_GRID)==360
# Policies may share the same physical candidate (e.g. adaptive=global). This is logged as
# a shared primitive, never represented as two independent GPU samples or charged twice in resources.
def heldout_batch(start,count=8):
    for pid,pol,s in TEST_GRID[start:start+count]:
        r=heldout_one(pid,pol,s)
        print(pid,pol,r['action'],s,'SUCCESS',r['evaluation']['success'],'LTP',r['loop_token_passes'])
    print('Heldout logical outcomes',len(all_recs('heldout')),'/',len(TEST_GRID),
          '| unique physical GPU candidates',len(all_recs('test_primitives')))

In [ ]:
heldout_batch(0,5)

In [ ]:
heldout_batch(5,5)

In [ ]:
heldout_batch(10,5)

In [ ]:
heldout_batch(15,5)

In [ ]:
heldout_batch(20,5)

In [ ]:
heldout_batch(25,5)

In [ ]:
heldout_batch(30,5)

In [ ]:
heldout_batch(35,5)

In [ ]:
heldout_batch(40,5)

In [ ]:
heldout_batch(45,5)

In [ ]:
heldout_batch(50,5)

In [ ]:
heldout_batch(55,5)

In [ ]:
heldout_batch(60,5)

In [ ]:
heldout_batch(65,5)

In [ ]:
heldout_batch(70,5)

In [ ]:
heldout_batch(75,5)

In [ ]:
heldout_batch(80,5)

In [ ]:
heldout_batch(85,5)

In [ ]:
heldout_batch(90,5)

In [ ]:
heldout_batch(95,5)

In [ ]:
heldout_batch(100,5)

In [ ]:
heldout_batch(105,5)

In [ ]:
heldout_batch(110,5)

In [ ]:
heldout_batch(115,5)

In [ ]:
heldout_batch(120,5)

In [ ]:
heldout_batch(125,5)

In [ ]:
heldout_batch(130,5)

In [ ]:
heldout_batch(135,5)

In [ ]:
heldout_batch(140,5)

In [ ]:
heldout_batch(145,5)

In [ ]:
heldout_batch(150,5)

In [ ]:
heldout_batch(155,5)

In [ ]:
heldout_batch(160,5)

In [ ]:
heldout_batch(165,5)

In [ ]:
heldout_batch(170,5)

In [ ]:
heldout_batch(175,5)

In [ ]:
heldout_batch(180,5)

In [ ]:
heldout_batch(185,5)

In [ ]:
heldout_batch(190,5)

In [ ]:
heldout_batch(195,5)

In [ ]:
heldout_batch(200,5)

In [ ]:
heldout_batch(205,5)

In [ ]:
heldout_batch(210,5)

In [ ]:
heldout_batch(215,5)

In [ ]:
heldout_batch(220,5)

In [ ]:
heldout_batch(225,5)

In [ ]:
heldout_batch(230,5)

In [ ]:
heldout_batch(235,5)

In [ ]:
heldout_batch(240,5)

In [ ]:
heldout_batch(245,5)

In [ ]:
heldout_batch(250,5)

In [ ]:
heldout_batch(255,5)

In [ ]:
heldout_batch(260,5)

In [ ]:
heldout_batch(265,5)

In [ ]:
heldout_batch(270,5)

In [ ]:
heldout_batch(275,5)

In [ ]:
heldout_batch(280,5)

In [ ]:
heldout_batch(285,5)

In [ ]:
heldout_batch(290,5)

In [ ]:
heldout_batch(295,5)

In [ ]:
heldout_batch(300,5)

In [ ]:
heldout_batch(305,5)

In [ ]:
heldout_batch(310,5)

In [ ]:
heldout_batch(315,5)

In [ ]:
heldout_batch(320,5)

In [ ]:
heldout_batch(325,5)

In [ ]:
heldout_batch(330,5)

In [ ]:
heldout_batch(335,5)

In [ ]:
heldout_batch(340,5)

In [ ]:
heldout_batch(345,5)

In [ ]:
heldout_batch(350,5)

In [ ]:
heldout_batch(355,5)

### B12. Two-attempt budget study: adaptive→global vs global→global
24 held-out problems × 2 arms × 2 seeds = 96 logical evaluations. Actual GPU calls are separately counted.

In [ ]:
STAGE_ARMS=('adaptive_then_global','global_then_global')
STAGE_GRID=[(pid,arm,s) for pid in COHORTS['heldout'] for arm in STAGE_ARMS for s in STAGED_SEEDS]
assert len(STAGE_GRID)==96

def staged_one(pid,arm,seed):
    tid=f'staged|{pid}|{arm}|{seed}'
    old=get_rec('staged',tid)
    if old:return old
    p,src=failure(pid)
    actions=(choose_action('adaptive',pid,seed),'global') if arm=='adaptive_then_global' else ('global','global')
    calls=[];accepted=None
    for j,action in enumerate(actions):
        # Distinct seeds across two attempts; no test-time gold answers involved in stopping.
        attempt_seed=seed+j*777
        primitive=repair_one(pid,action,attempt_seed,REPAIR_MODE,stage='staged_primitives')
        calls.append(primitive)
        if primitive['public_accept']:
            accepted=primitive;break
    # If neither meets the public gate, the last candidate is the output (not oracle-selected).
    selected=accepted if accepted else calls[-1]
    r={'trial_id':tid,'protocol_hash':RUN_SIG,'problem_id':pid,'arm':arm,'seed':seed,
       'actions_used':[x['action'] for x in calls],'proposal_count':len(calls),
       'proposal_ids':[x['trial_id'] for x in calls],
       'public_accept':selected['public_accept'],'evaluation':selected['evaluation'],
       'loop_token_passes':sum(x['loop_token_passes'] for x in calls),
       'wall_seconds':sum(x['wall_seconds'] for x in calls),
       'recurrent_passes':sum(x['recurrent_passes'] for x in calls)}
    save_rec('staged',r)
    return r

def staged_batch(start,count=6):
    for pid,arm,s in STAGE_GRID[start:start+count]:
        r=staged_one(pid,arm,s)
        print(pid,arm,s,'success',r['evaluation']['success'],'proposals',r['proposal_count'],
              'loop_token_passes',r['loop_token_passes'])
    print('Staged',len(all_recs('staged')),'/',len(STAGE_GRID))

In [ ]:
staged_batch(0,4)

In [ ]:
staged_batch(4,4)

In [ ]:
staged_batch(8,4)

In [ ]:
staged_batch(12,4)

In [ ]:
staged_batch(16,4)

In [ ]:
staged_batch(20,4)

In [ ]:
staged_batch(24,4)

In [ ]:
staged_batch(28,4)

In [ ]:
staged_batch(32,4)

In [ ]:
staged_batch(36,4)

In [ ]:
staged_batch(40,4)

In [ ]:
staged_batch(44,4)

In [ ]:
staged_batch(48,4)

In [ ]:
staged_batch(52,4)

In [ ]:
staged_batch(56,4)

In [ ]:
staged_batch(60,4)

In [ ]:
staged_batch(64,4)

In [ ]:
staged_batch(68,4)

In [ ]:
staged_batch(72,4)

In [ ]:
staged_batch(76,4)

In [ ]:
staged_batch(80,4)

In [ ]:
staged_batch(84,4)

In [ ]:
staged_batch(88,4)

In [ ]:
staged_batch(92,4)

### B13. Final analysis: exact trials, problem-level sign tests, compute accounting, recurrence and format compliance
The main conclusion uses held-out *distinct problems*, not repeated trial counts as independent evidence. Exploratory multiplicity is declared; no selective claiming of wins.

In [ ]:

from scipy.stats import beta,binomtest
import itertools

def exact_binomial_ci(k,n,alpha=.05):
    return ((0.0 if k==0 else float(beta.ppf(alpha/2,k,n-k+1))),
            (1.0 if k==n else float(beta.ppf(1-alpha/2,k+1,n-k))))

def summarize(rows,field):
    out=[]
    for arm in sorted(set(r[field] for r in rows)):
        rr=[r for r in rows if r[field]==arm]
        success=[r for r in rr if r['evaluation']['success']]
        distinct={r['problem_id'] for r in success}
        nproblem=len({r['problem_id'] for r in rr});pn=len(distinct)
        lo,hi=exact_binomial_ci(pn,nproblem)
        out.append({field:arm,'n_trials':len(rr),'success_trials':len(success),
                   'distinct_problems':nproblem,'problems_recovered':pn,
                   'exact95_problem_low':lo,'exact95_problem_high':hi,
                   'strict_format_successes':sum(bool(r['evaluation']['strict_correct']) for r in rr),
                   'mean_loop_token_passes':statistics.mean(r['loop_token_passes'] for r in rr),
                   'mean_wall_seconds':statistics.mean(r['wall_seconds'] for r in rr)})
    return out

def paired_problem_comparison(records,a,b,key='policy'):
    ids=sorted({r['problem_id'] for r in records})
    table=[]
    for pid in ids:
        aa=[r for r in records if r['problem_id']==pid and r[key]==a]
        bb=[r for r in records if r['problem_id']==pid and r[key]==b]
        assert aa and bb and {r['seed'] for r in aa}=={r['seed'] for r in bb}, 'Unpaired seeds or missing arm'
        table.append((pid,int(any(r['evaluation']['success'] for r in aa)),int(any(r['evaluation']['success'] for r in bb))))
    wins=sum(x>y for _,x,y in table);losses=sum(x<y for _,x,y in table)
    p=binomtest(wins,wins+losses,.5).pvalue if wins+losses else 1.0
    return {'A':a,'B':b,'n_independent_problems':len(table),'A_only_problems':wins,
            'B_only_problems':losses,'both_or_neither':len(table)-wins-losses,
            'difference_in_problem_coverage':(wins-losses)/len(table),'exact_two_sided_p':float(p)}

required={'calibration':96,'heldout':360,'staged':96}
counts={s:len(all_recs(s)) for s in required}
assert counts==required, 'Incomplete experiment; do not freeze partial journal tables: '+str(counts)
assert len(COHORTS['calibration'])==12 and len(COHORTS['heldout'])==24
assert len(set(COHORTS['calibration']))==12 and len(set(COHORTS['heldout']))==24
assert not (set(COHORTS['calibration'])&set(COHORTS['heldout']))
for pid in COHORTS['calibration']+COHORTS['heldout']:
    p,src=failure(pid)
    assert src['evaluation']['eligible_semantic_failure']
    assert src['evaluation']['failure_type']=='definite_wrong'
for stage in required:
    for r in all_recs(stage):assert r['protocol_hash']==RUN_SIG
for st in ('calibration','test_primitives','staged_primitives'):
    for r in all_recs(st):assert r['exact_prefix_preserved']
rows=all_recs('heldout'); two=all_recs('staged')
H=pd.DataFrame(summarize(rows,'policy'))
S=pd.DataFrame(summarize(two,'arm'))
CAL=pd.DataFrame(summarize(all_recs('calibration'),'action'))
PAIR=pd.DataFrame([paired_problem_comparison(rows,'adaptive',other) for other in ('global','best_fixed','random_scope','tail50')]
                  +[paired_problem_comparison(two,'adaptive_then_global','global_then_global',key='arm')])
ART.mkdir(parents=True,exist_ok=True)
for name,frame in [('primary_heldout_policy_summary',H),('two_stage_summary',S),('calibration_summary',CAL),('paired_problem_tests',PAIR)]:
    frame.to_csv(ART/(name+'.csv'),index=False)

# Distinct physical calls counted once, even where policies share a primitive.
compute={'physical_single_trial_proposals':len(all_recs('test_primitives')),
         'logical_heldout_policy_evaluations':len(rows),
         'physical_two_stage_proposals':len(all_recs('staged_primitives')),
         'logical_two_stage_evaluations':len(two),
         'original_semantic_failures':{'calibration':12,'heldout':24},
         'base_semantic_failure_counts':Counter(r['kind'] for r in all_recs('natural') if r['evaluation']['eligible_semantic_failure']),
         'base_format_only_or_unanswered_counts':Counter(r['evaluation']['failure_type'] for r in all_recs('natural'))}
atomic_write(ART/'compute_and_eligibility_audit.json',compute)

fig,axes=plt.subplots(figsize=(9,4.4))
Hsort=H.sort_values('policy')
axes.bar(Hsort['policy'],Hsort['problems_recovered'])
axes.set_ylabel('Distinct held-out semantic errors recovered (out of 24)')
axes.set_title('Repair policy coverage; repeated seeds clustered by problem')
plt.xticks(rotation=18,ha='right');plt.tight_layout()
fig.savefig(ART/'figure_semantic_problem_recovery.png',dpi=175)
fig.savefig(ART/'figure_semantic_problem_recovery.pdf');plt.close(fig)
fig,ax=plt.subplots(figsize=(8.4,4.5))
for _,r in H.iterrows():
    ax.scatter(r['mean_loop_token_passes'],r['problems_recovered'],s=70)
    ax.annotate(r['policy'],(r['mean_loop_token_passes'],r['problems_recovered']),xytext=(4,5),textcoords='offset points')
ax.set_xlabel('Mean native loop-token passes / policy trial')
ax.set_ylabel('Distinct held-out problems recovered')
ax.set_title('Measured compute and recovery (not assumed compute-matched)');plt.tight_layout()
fig.savefig(ART/'figure_compute_recovery.png',dpi=175)
fig.savefig(ART/'figure_compute_recovery.pdf');plt.close(fig)

# Holm multiplicity procedure for four primary exploratory contrasts.
P=PAIR.iloc[:4].copy().sort_values('exact_two_sided_p').reset_index(drop=True)
P['holm_adjusted_p']=1.0
v=0.0
for i in range(len(P)):
    v=max(v,float(P.loc[i,'exact_two_sided_p'])*(len(P)-i))
    P.loc[i,'holm_adjusted_p']=min(1.0,v)
P.to_csv(ART/'primary_paired_with_holm.csv',index=False)

MAN={'protocol_hash':RUN_SIG,'fixed_bank_sha256':BANK_HASH,'frozen_policy_sha256':sha(POLICY),
     'status':'COMPLETE','stages':counts,'primary_metric':'distinct_semantically_wrong problems recovered',
     'limits':['Synthetic benchmark; independent external validation remains necessary',
               'Human audit of a subset recommended; parser may miss unconventional answer formats',
               'Model results may be negative; null results must be reported',
               'Trial evaluations reuse proposal outputs and are not independent model generations']}
atomic_write(ART/'research_manifest.json',MAN)
print(H[['policy','success_trials','problems_recovered','mean_loop_token_passes']].to_string(index=False))
print('\nPaired problem tests:\n',PAIR.to_string(index=False))
print('NB29 final analysis COMPLETE. Figures and tables in',ART)


### B14. Export current state (SAFE to run at ANY point, including a partial run)
If the semantic-failure quota is not met, export the partial archive and resume next session. **Partial results are never labelled as complete.**

In [ ]:

def export_nb29():
    dst=Path('/content/RepairScopeX_NB29_Semantic_Repair_results.zip')
    optional=[ROOT/'protocol.json',ROOT/'frozen_problem_bank.json',ROOT/'frozen_cohorts.json',ROOT/'FROZEN_POLICY.json']
    items=[]
    for p in optional:
        if p.is_file():items.append((p,'protocol/'+p.name))
    if ART.exists():
        for p in ART.rglob('*'):
            if p.is_file():items.append((p,'artifacts/'+str(p.relative_to(ART))))
    for p in CACHE.rglob('*.json'):
        items.append((p,'records/'+str(p.relative_to(CACHE))))
    # NB21 has already finished and is analyzed separately by Notebook 30.
    assert len(set(name for _,name in items))==len(items)
    seen={name:hashlib.sha256(p.read_bytes()).hexdigest() for p,name in items}
    stages={s:len(all_recs(s)) for s in ('natural','calibration','heldout','test_primitives','staged','staged_primitives')}
    needed={'calibration':96,'heldout':360,'staged':96}
    is_complete=(all(stages[s]==v for s,v in needed.items()) and
                 (ART/'research_manifest.json').exists())
    details={'protocol_hash':RUN_SIG,'semantic_eligibility':'explicitly identifiable WRONG answers only',
             'partial':not is_complete,'stage_counts':stages,'required':needed,
             'nb21_included':False,'sha256':seen}
    with zipfile.ZipFile(dst,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        z.writestr('EXPORT_STATUS.json',json.dumps(details,indent=2,default=str))
        for p,name in sorted(items,key=lambda t:t[1]):z.write(p,name)
    with zipfile.ZipFile(dst) as z:
        assert z.testzip() is None
        for name,digest in seen.items():assert hashlib.sha256(z.read(name)).hexdigest()==digest
    print('Export',dst,'partial?',not is_complete,'saved stages',stages,'NB21 included',False)
    return dst
NB29_ZIP=export_nb29()
from google.colab import files
files.download(str(NB29_ZIP))


### Final research interpretation and sharing

Return the combined ZIP (`RepairScopeX_NB29_Semantic_Repair_results.zip`), plus the MDLM NB21 ZIP if you exported that separately. 

**Strict separation:** NB21's 64 controlled MDLM infills and NB29's ALoDLM semantic-failure trials belong to different architectures, tasks, precisions, and protocols. Never pool trial-level success rates as if exchangeable.

A positive adaptive result is publishable only after verification of semantic correctness, independent problem-level paired comparison, and honest inference-compute accounting. A null outcome is also a valid result and should not be hidden. External held-out data and additional model transfer are not included in the synthetic NB29 phase; they remain independent confirmation work.


### Next: Notebook 30 (CPU)
Upload the exported `RepairScopeX_NB29_Semantic_Repair_results.zip` to Notebook 30 for checksum validation, pairing checks, and integration with the completed NB21 cohort. Notebook 30 does **not** turn partial NB29 experiments into publication results.